# 00 · CONFIG — Inicialização dos Schemas Delta

**Pipeline:** ETL IPCA × Boi Gordo · Medallion Architecture  
**Execução:** Rodar UMA VEZ antes de qualquer outro notebook.  
**Ambiente:** Databricks Community Edition (Serverless)

---
**Ordem de execução obrigatória:**
```
00.config → 01.bronze/ipca → 01.bronze/boi_gordo → 02.silver/economia → 03.gold/insights
```

In [ ]:
# ============================================================
# CELL 1 — Verificar sessão Spark
# ============================================================
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print("Sessão ativa  : OK")

In [ ]:
# ============================================================
# CELL 2 — Criar databases (Databricks CE · DBFS)
# NOTA: No CE sem Unity Catalog usamos dbfs:/FileStore/
#       como LOCATION para garantir persistência.
# ============================================================

spark.sql("""
    CREATE DATABASE IF NOT EXISTS etl_pos
    COMMENT 'Namespace raiz do projeto ETL IPCA x Boi Gordo'
""")

spark.sql("""
    CREATE DATABASE IF NOT EXISTS etl_pos__bronze
    LOCATION 'dbfs:/FileStore/etl_pos/bronze/'
    COMMENT 'Camada Bronze - dados brutos ingeridos'
""")

spark.sql("""
    CREATE DATABASE IF NOT EXISTS etl_pos__silver
    LOCATION 'dbfs:/FileStore/etl_pos/silver/'
    COMMENT 'Camada Silver - dados limpos e normalizados'
""")

spark.sql("""
    CREATE DATABASE IF NOT EXISTS etl_pos__gold
    LOCATION 'dbfs:/FileStore/etl_pos/gold/'
    COMMENT 'Camada Gold - insights e metricas calculadas'
""")

print("Databases criados com sucesso!")

In [ ]:
# ============================================================
# CELL 3 — Validar criação
# ============================================================
print("=" * 50)
print("DATABASES DISPONÍVEIS:")
print("=" * 50)
spark.sql("SHOW DATABASES LIKE 'etl_pos*'").show(truncate=False)

# Verificar diretórios no DBFS
import subprocess
for layer in ['bronze', 'silver', 'gold']:
    try:
        dbutils.fs.mkdirs(f"dbfs:/FileStore/etl_pos/{layer}/")
        print(f"  dbfs:/FileStore/etl_pos/{layer}/ — OK")
    except Exception as e:
        print(f"  {layer}: {e}")

In [ ]:
# ============================================================
# CELL 4 — Variáveis de sessão globais
# ATENÇÃO: Re-executar este notebook se cluster reiniciar!
# ============================================================
DB_BRONZE = "etl_pos__bronze"
DB_SILVER = "etl_pos__silver"
DB_GOLD   = "etl_pos__gold"

TBL_BRONZE_IPCA     = f"{DB_BRONZE}.ipca"
TBL_BRONZE_BOI      = f"{DB_BRONZE}.boi_gordo"
TBL_SILVER_ECONOMIA = f"{DB_SILVER}.economia"
TBL_GOLD_INSIGHTS   = f"{DB_GOLD}.indicadores"

print("Variáveis de sessão definidas:")
for var in [TBL_BRONZE_IPCA, TBL_BRONZE_BOI, TBL_SILVER_ECONOMIA, TBL_GOLD_INSIGHTS]:
    print(f"  {var}")